# Phase 3: Optimized GPU Implementation (Tiled Convolution)
**CSC14120 - Parallel Programming**

---

## 3.1 Mục tiêu Phase 3

Tối ưu hóa các kernel CUDA để đạt hiệu suất cao hơn Phase 2 Naive:
- **Shared Memory Tiling** cho Convolution
- **Vectorized Memory Access** (float4)
- **Loop Unrolling** với `#pragma unroll`
- **Kernel Fusion** (Conv + ReLU)

## 3.2 Kỹ thuật tối ưu được áp dụng

### A. Shared Memory Tiling (Category 1: Memory Optimization)

```cpp
// Tiled Convolution - Load input tiles to shared memory
__global__ void conv2d_forward_tiled_kernel(...) {
    extern __shared__ float shared_mem[];
    float* s_input = shared_mem;
    
    // Cooperative loading of input tile
    for (int ic = 0; ic < in_c; ++ic) {
        // Load tile to shared memory
        for (int load = 0; load < num_loads; ++load) {
            int linear_idx = load * threads_per_block + linear_tid;
            if (linear_idx < tile_size) {
                s_input[linear_idx] = input[...];
            }
        }
        __syncthreads();
        
        // Compute using shared memory (fast!)
        for (int kh = 0; kh < k; ++kh) {
            for (int kw = 0; kw < k; ++kw) {
                sum += s_input[...] * weights[...];
            }
        }
        __syncthreads();
    }
}
```

### B. Vectorized Memory Access (Category 2: Kernel-Level)

```cpp
// ReLU với float4 - xử lý 4 phần tử cùng lúc
__global__ void relu_forward_vectorized_kernel(const float4* input, float4* output, size_t n4) {
    size_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n4) {
        float4 in = input[idx];
        float4 out;
        out.x = fmaxf(0.0f, in.x);
        out.y = fmaxf(0.0f, in.y);
        out.z = fmaxf(0.0f, in.z);
        out.w = fmaxf(0.0f, in.w);
        output[idx] = out;
    }
}
```

### C. Loop Unrolling

```cpp
#pragma unroll
for (int kh = 0; kh < 3; ++kh) {
    #pragma unroll
    for (int kw = 0; kw < 3; ++kw) {
        // Compiler sẽ unroll hoàn toàn vòng lặp 3x3
    }
}
```

### D. Các tối ưu khác

| Kỹ thuật | Mô tả | Expected Speedup |
|:---------|:------|:-----------------|
| Pinned Memory | `cudaMallocHost` cho faster transfers | 1.5-2x |
| CUDA Streams | Double buffering, overlap transfer/compute | 1.2-1.5x |
| Fast Math | `--use_fast_math` compiler flag | 1.1-1.2x |
| He Initialization | Proper weight init for ReLU | Better convergence |

---

## Hướng dẫn chạy:
1. Zip thư mục project (không bao gồm data/)
2. Upload file zip lên Colab
3. Chạy tất cả cells

In [ ]:
# Kiểm tra GPU
!nvidia-smi
!nvcc --version

In [ ]:
# Upload và giải nén project
from google.colab import files
import zipfile, os

print("Upload file zip project:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('project')

for root, dirs, _ in os.walk('project'):
    if 'src' in dirs:
        %cd {root}
        break
!ls

In [ ]:
# Download CIFAR-10
import urllib.request, tarfile
os.makedirs('data', exist_ok=True)

if not os.path.exists('data/data_batch_1.bin'):
    print('Downloading CIFAR-10...')
    urllib.request.urlretrieve('https://www.cs.toronto.edu/~kriz/cifar-10-binary.tar.gz', 'data/cifar.tar.gz')
    with tarfile.open('data/cifar.tar.gz', 'r:gz') as tar:
        tar.extractall('data')
    !mv data/cifar-10-batches-bin/* data/
print('Done!')

## 3.3 Build Phase 3 (Optimized - Tiled Convolution)

**Compiler flags:**
- `-O3`: Maximum optimization
- `-DUSE_OPTIMIZED_KERNELS`: Enable optimized kernel path
- `--use_fast_math`: Fast math operations
- `-Xptxas -O3`: PTX assembler optimization
- `--maxrregcount=64`: Limit registers for better occupancy
- `-lcublas`: cuBLAS for GEMM-based convolution
- `-lcurand`: cuRAND for He Initialization

**Lưu ý:** Version này sử dụng Tiled Convolution tự viết (không dùng cuDNN)

In [ ]:
# Build Phase 3 với Tiled Convolution (không dùng cuDNN)
# Sử dụng -DUSE_TILED_CONV thay vì cuDNN

!nvcc -O3 -std=c++17 -arch=sm_75 --expt-relaxed-constexpr \
    --use_fast_math -Xptxas -O3 --maxrregcount=64 \
    -DUSE_OPTIMIZED_KERNELS -DUSE_TILED_CONV \
    -Iinclude -lcublas -lcurand \
    -o gpu_train_tiled \
    src/main_gpu.cu src/layers_gpu.cu src/gpu_autoencoder.cu src/layers_gpu_opt.cu src/dataset.cpp

print('Build complete with Tiled Convolution + cuBLAS!')

## 3.4 Training

**Expected performance (Phase 3 Tiled):**
- Time per epoch: ~30-60 seconds
- Total training: ~10-20 minutes
- Speedup vs Phase 2: ~3-5x

In [ ]:
# Train với Tiled Convolution
!./gpu_train_tiled --data data --epochs 20 --batch 64 --lr 0.001 \
    --log phase3_tiled.csv --log-txt phase3_tiled.txt --save-weights phase3_tiled.weights

## 3.5 Kết quả và Visualization

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('phase3_tiled.csv')
ep = df[df['batch'].isna()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep['epoch'], ep['loss'], 'r-o')
ax1.set_title('Training Loss per Epoch')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True)

ax2.plot(ep['epoch'], ep['epoch_time_sec'], 'orange', marker='o')
ax2.set_title('Time per Epoch')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Time (s)')
ax2.grid(True)

plt.tight_layout()
plt.savefig('phase3_tiled_results.png', dpi=150)
plt.show()

print("="*50)
print("PHASE 3 TILED CONVOLUTION RESULTS")
print("="*50)
print(f"Best Loss: {ep['best_loss'].iloc[-1]:.6f}")
print(f"Final Loss: {ep['loss'].iloc[-1]:.6f}")
print(f"Avg Time per Epoch: {ep['epoch_time_sec'].mean():.2f}s")
print(f"Total Training Time: {ep['epoch_time_sec'].sum():.2f}s ({ep['epoch_time_sec'].sum()/60:.2f} min)")
print("="*50)

In [ ]:
# Show training log
print("="*60)
print("TRAINING LOG (last 30 lines)")
print("="*60)
!tail -30 phase3_tiled.txt

In [ ]:
# Download results
from google.colab import files
files.download('phase3_tiled.csv')
files.download('phase3_tiled.txt')
files.download('phase3_tiled.weights')
files.download('phase3_tiled_results.png')

## 3.6 So sánh Performance Phase 2 vs Phase 3

| Metric | Phase 2 (Naive) | Phase 3 (Tiled) | Speedup |
|:-------|:----------------|:----------------|:--------|
| Time/Epoch | ~120-300s | ~30-60s | 3-5x |
| Memory Access | Global only | Shared + Global | - |
| Data Reuse | None | Tile reuse | - |
| Vectorization | No | float4 for ReLU | - |

### Tại sao Tiled Convolution nhanh hơn?

1. **Shared Memory**: Latency ~5 cycles vs Global Memory ~400-800 cycles
2. **Data Reuse**: Mỗi input pixel được load 1 lần, dùng nhiều lần
3. **Coalesced Access**: Threads trong cùng warp truy cập bộ nhớ liên tiếp
4. **Reduced Bandwidth**: Giảm traffic đến global memory